# Audio Preprocessing Pipeline

This notebook prepares a clean training-ready dataset from the raw `AudioWAV/` folder.

It does four things:
1. Loads the raw manifest.
2. Excludes the known exact-duplicate file pairs from the audit report.
3. Standardizes each kept clip and writes it to `data/processed_audio/`.
4. Writes a processed manifest to `manifests/processed/processed_audio_manifest.csv`.

This version avoids external dependencies like `librosa` and `soundfile`, so it runs in your current environment.

In [ ]:
from __future__ import annotations

import wave
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from tqdm.auto import tqdm
except Exception:
    def tqdm(iterable, total=None):
        return iterable


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "AudioWAV").exists() and (candidate / "context.md").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing AudioWAV/ and context.md")


ROOT = find_repo_root(Path.cwd().resolve())
RAW_MANIFEST = ROOT / "manifests" / "audio_manifest.csv"
OUTPUT_AUDIO_DIR = ROOT / "data" / "processed_audio"
OUTPUT_MANIFEST = ROOT / "manifests" / "processed" / "processed_audio_manifest.csv"
OUTPUT_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_MANIFEST.parent.mkdir(parents=True, exist_ok=True)

TARGET_SAMPLE_RATE = 16_000
TARGET_PEAK = 0.98
MAX_WORKERS = 8

# Conservative policy: exclude both sides of each conflicting exact-duplicate pair
# until labels are manually reviewed.
EXCLUDED_DUPLICATE_FILE_NAMES = {
    "1006_TIE_HAP_XX.wav",
    "1006_TIE_NEU_XX.wav",
    "1013_WSI_DIS_XX.wav",
    "1013_WSI_SAD_XX.wav",
    "1017_IWW_ANG_XX.wav",
    "1017_IWW_FEA_XX.wav",
}

ROOT

## Load and filter the manifest

The dataset audit confirmed the raw files are already mono 16 kHz 16-bit PCM. That means preprocessing can stay simple and robust.

In [ ]:
manifest = pd.read_csv(RAW_MANIFEST)
manifest["file_path"] = manifest["file_path"].map(lambda p: str(Path(p)))
manifest["is_excluded_duplicate"] = manifest["file_name"].isin(EXCLUDED_DUPLICATE_FILE_NAMES)

processed_manifest = manifest.loc[~manifest["is_excluded_duplicate"]].copy()
processed_manifest["source_file_path"] = processed_manifest["file_path"]
processed_manifest["processed_file_path"] = processed_manifest["clip_id"].map(
    lambda clip_id: str(OUTPUT_AUDIO_DIR / f"{clip_id}.wav")
)

print(f"Raw rows: {len(manifest):,}")
print(f"Excluded duplicate rows: {manifest['is_excluded_duplicate'].sum():,}")
print(f"Rows to process: {len(processed_manifest):,}")
processed_manifest[["split", "emotion"]].value_counts().sort_index().head(12)

## Standardize audio

For this dataset the preprocessing intentionally stays light:
- read PCM WAV
- ensure mono 16 kHz input
- remove DC offset
- apply peak normalization
- write clean 16-bit PCM WAV

This does not aggressively denoise or trim silence, because that can remove emotional cues.

In [ ]:
def read_pcm16_mono_wav(path: str) -> tuple[np.ndarray, int]:
    with wave.open(str(path), "rb") as wav_file:
        channels = wav_file.getnchannels()
        sample_rate = wav_file.getframerate()
        sample_width = wav_file.getsampwidth()
        n_frames = wav_file.getnframes()
        compression = wav_file.getcomptype()
        raw_bytes = wav_file.readframes(n_frames)

    if compression != "NONE":
        raise ValueError(f"Unsupported compressed WAV: {compression}")
    if sample_width != 2:
        raise ValueError(f"Expected 16-bit PCM WAV, found sample width {sample_width}")
    if sample_rate != TARGET_SAMPLE_RATE:
        raise ValueError(f"Expected {TARGET_SAMPLE_RATE} Hz audio, found {sample_rate} Hz")

    audio = np.frombuffer(raw_bytes, dtype="<i2").astype(np.float32)
    if channels > 1:
        audio = audio.reshape(-1, channels).mean(axis=1)
    audio = audio / 32768.0
    return audio, sample_rate


def write_pcm16_mono_wav(path: str, audio: np.ndarray, sample_rate: int) -> None:
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    audio = np.asarray(audio, dtype=np.float32)
    audio = np.clip(audio, -1.0, 1.0)
    pcm = (audio * 32767.0).astype("<i2")
    with wave.open(str(target), "wb") as wav_file:
        wav_file.setnchannels(1)
        wav_file.setsampwidth(2)
        wav_file.setframerate(sample_rate)
        wav_file.writeframes(pcm.tobytes())


def standardize_audio(source_path: str, target_path: str) -> dict:
    audio, sample_rate = read_pcm16_mono_wav(source_path)
    audio = audio - float(np.mean(audio))
    peak = float(np.max(np.abs(audio))) if audio.size else 0.0
    if peak > 0:
        audio = (audio / peak) * TARGET_PEAK
    write_pcm16_mono_wav(target_path, audio, sample_rate)
    return {
        "processed_sample_rate": sample_rate,
        "processed_num_samples": int(audio.shape[0]),
        "processed_duration_sec": round(audio.shape[0] / float(sample_rate), 6),
        "status": "ok",
    }


def process_row(row: dict) -> dict:
    try:
        return {
            "clip_id": row["clip_id"],
            **standardize_audio(row["source_file_path"], row["processed_file_path"]),
        }
    except Exception as exc:  # noqa: BLE001
        return {
            "clip_id": row["clip_id"],
            "processed_sample_rate": np.nan,
            "processed_num_samples": np.nan,
            "processed_duration_sec": np.nan,
            "status": f"error: {exc}",
        }


records = processed_manifest.to_dict(orient="records")
results = []

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    for result in tqdm(executor.map(process_row, records), total=len(records)):
        results.append(result)

results_df = pd.DataFrame(results)
results_df["status"].value_counts()

In [ ]:
results_df = results_df.set_index("clip_id")
processed_manifest = processed_manifest.join(results_df, on="clip_id", rsuffix="_result")

required_columns = {"processed_file_path", "processed_duration_sec", "status"}
missing_columns = required_columns - set(processed_manifest.columns)
if missing_columns:
    raise KeyError(f"Missing expected columns: {sorted(missing_columns)}")

processed_manifest = processed_manifest.loc[processed_manifest["status"] == "ok"].copy()
processed_manifest.to_csv(OUTPUT_MANIFEST, index=False)

summary = {
    "processed_rows": len(processed_manifest),
    "excluded_duplicate_rows": int(manifest["is_excluded_duplicate"].sum()),
    "failed_rows": int((results_df["status"] != "ok").sum()),
    "output_manifest": str(OUTPUT_MANIFEST),
    "output_audio_dir": str(OUTPUT_AUDIO_DIR),
}
summary

In [ ]:
display_columns = [
    "clip_id",
    "emotion",
    "split",
    "source_file_path",
    "processed_file_path",
    "processed_duration_sec",
]
processed_manifest[display_columns].head()

In [ ]:
existing_processed_files = sum(Path(path).exists() for path in processed_manifest["processed_file_path"])
print(f"Processed manifest rows: {len(processed_manifest):,}")
print(f"Processed audio files present: {existing_processed_files:,}")
print(f"All output files written: {existing_processed_files == len(processed_manifest)}")